# Claude Agent SDK 测试 (Notebook 适配版)

**环境**: Python 3.13.9 (base) | `claude-agent-sdk 0.2.103` | `deepseek-v4-pro`

**关键解决**: Agent SDK 通过 `anyio.open_process` 启动 CLI 子进程。
Jupyter 的嵌套事件循环在 Windows 上与 `anyio` 不兼容(`NotImplementedError`)。
因此将 SDK 调用封装在独立线程中,用 `asyncio.run()` 创建干净的事件循环。

**用法**: 只需运行 Cell 0,然后直接调用 `run_agent(prompt, ...)`。

## 0. 配置注入 + run_agent 封装

这个 cell 做了三件事:
1. 从 `~/.claude/settings.json` 自动读取 API 配置
2. 在线程中运行 Agent SDK(避开 Jupyter 事件循环冲突)
3. 提供 `run_agent()` 函数供后续 cell 使用

In [ ]:
'''
Cell 0: 配置注入 + Agent SDK 线程封装

运行此 cell 后,后续所有 cell 都可以直接调用 run_agent()。
'''
import os, json, asyncio, threading, sys
from pathlib import Path
from claude_agent_sdk import (
    query, ClaudeAgentOptions, create_sdk_mcp_server,
)
from claude_agent_sdk import (
    AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,
)


# ============================================================
# 1. 配置注入
# ============================================================

def inject_config():
    '''
    从 ~/.claude/settings.json 读取 API 配置,注入 os.environ。
    缺失的变量用默认值补充。
    '''
    settings_path = Path.home() / ".claude" / "settings.json"
    if settings_path.exists():
        with open(settings_path, encoding="utf-8") as f:
            env_vars = json.load(f).get("env", {})
        for key, value in env_vars.items():
            if key.startswith("ANTHROPIC") or key.startswith("CLAUDE_CODE"):
                os.environ[key] = os.path.expandvars(value)

    defaults = {
        "ANTHROPIC_BASE_URL": "http://192.168.1.67:9443/anthropic",
        "ANTHROPIC_AUTH_TOKEN": "c5930172761e6415e2d2e1ddc5f74108",
        "ANTHROPIC_MODEL": "deepseek-v4-pro",
    }
    for k, v in defaults.items():
        if k not in os.environ:
            os.environ[k] = v

inject_config()
print("配置注入完成:")
for v in ["ANTHROPIC_BASE_URL", "ANTHROPIC_AUTH_TOKEN", "ANTHROPIC_MODEL"]:
    print(f"  [OK] {v}")


# ============================================================
# 2. run_agent: 线程中运行 Agent SDK
# ============================================================

# 项目根目录：脚本/notebook 所在目录
PROJECT_ROOT = os.getcwd()
# SDK sessions 隔离目录：与手动使用 Claude Code 的 session 互不可见
SDK_SESSIONS_DIR = os.path.join(PROJECT_ROOT, ".sdk_sessions")

# 默认禁用的工具：默认全部放行（Write/Edit 受限于 cwd，add_dirs 天然只读）
DEFAULT_DISALLOWED_TOOLS = []

def run_agent(
    prompt: str,
    *,
    model: str = "deepseek-v4-pro",
    system_prompt: str = "",
    max_turns: int | None = None,
    allowed_tools: list[str] | None = None,
    disallowed_tools: list[str] | None = None,
    permission_mode: str = "bypassPermissions",
    tools: list[str] | None = None,
    cwd: str | None = None,
    add_dirs: list[str] | None = None,
    mcp_servers: dict | None = None,
    effort: str = "max",
    skills: list[str] | str | None = None,
    plugins: list[dict] | None = None,
    thinking: dict | None = None,
) -> dict:
    '''
    在独立线程中运行 Agent SDK 查询。

    Agent SDK 内部使用 anyio.open_process 启动 CLI 子进程。
    Jupyter 的嵌套事件循环与 anyio 在 Windows 上不兼容。
    在独立线程中用 asyncio.run() 创建全新事件循环避开。

    默认行为:
    - cwd = 项目根目录下的 .sdk_sessions/（session 与手动 Claude Code 隔离）
    - add_dirs = 项目根目录（保留对项目文件的完整访问权）
    - disallowed_tools = []（全部放行，写入受限于 cwd，add_dirs 只读）
    - effort = "max"（最高推理力度）

    Args:
        prompt: 用户提问
        model: 模型名称
        system_prompt: 系统提示
        max_turns: 最大对话轮次,None=不限制
        allowed_tools: 额外自动许可的工具列表
        disallowed_tools: 禁用的工具列表,默认 []（全部放行，写入受限于 cwd）
        permission_mode: 权限模式,默认 "bypassPermissions"
        tools: 模型可用的工具集,None=全部内置工具
        cwd: 工作目录（默认 .sdk_sessions/）
        add_dirs: 额外可访问的目录（默认项目根目录）
        mcp_servers: MCP 工具服务器配置
        effort: 推理力度,默认 "max"
        skills: 启用的 skill,"all"=全部加载
        plugins: 本地插件列表
        thinking: 思考模式

    Returns:
        {text, stop_reason, duration_ms, num_turns}
    '''

    # 默认值：cwd → 隔离目录，add_dirs → 项目根目录
    if cwd is None:
        cwd = SDK_SESSIONS_DIR
        os.makedirs(cwd, exist_ok=True)  # 确保目录存在
    if add_dirs is None:
        add_dirs = [PROJECT_ROOT]
    if disallowed_tools is None:
        disallowed_tools = DEFAULT_DISALLOWED_TOOLS

    async def _run():
        options = ClaudeAgentOptions(
            model=model,
            permission_mode=permission_mode,
            max_turns=max_turns,
            allowed_tools=allowed_tools or [],
            disallowed_tools=disallowed_tools,
            tools=tools,
            cwd=cwd,
            add_dirs=add_dirs,
            system_prompt=system_prompt or None,
            mcp_servers=mcp_servers or {},
            effort=effort,
            skills=skills,
            plugins=plugins or [],
            thinking=thinking,
        )

        text_parts = []
        stats = {}

        async for msg in query(prompt=prompt, options=options):
            if isinstance(msg, AssistantMessage):
                for block in msg.content:
                    if isinstance(block, TextBlock):
                        # 安全打印(避免 GBK 编码问题)
                        try:
                            print(block.text, end="", flush=True)
                        except UnicodeEncodeError:
                            print(block.text.encode("ascii", "replace").decode(), end="")
                        text_parts.append(block.text)
                    elif isinstance(block, ToolUseBlock):
                        print("\n[call: {}]".format(block.name), end="")
            elif isinstance(msg, ResultMessage):
                stats = {
                    "stop_reason": msg.stop_reason or "unknown",
                    "duration_ms": msg.duration_ms,
                    "num_turns": msg.num_turns,
                    "cost": msg.total_cost_usd,
                    "is_error": msg.is_error,
                }

        print("\n--- [{}] {}ms | {} turns ---".format(
            stats.get("stop_reason", "?"),
            stats.get("duration_ms", 0),
            stats.get("num_turns", 0),
        ))
        return {"text": "".join(text_parts), **stats}

    return asyncio.run(_run())

print(f"项目根目录: {PROJECT_ROOT}")
print(f"SDK Session 目录: {SDK_SESSIONS_DIR}")
print("run_agent() 就绪 -- 可以直接调用了")


## 1. 简单查询

In [ ]:
result = run_agent(
    prompt="请用一句话介绍你自己",
    max_turns=3,
)

## 2. 系统提示

In [ ]:
result = run_agent(
    prompt="夏普比率和最大回撤哪个更重要?简要说明。",
    system_prompt="你是量化金融分析师。用中文回答,简洁专业。",
    max_turns=3,
)

## 3. 内置工具与权限

**默认策略：全部放行。Write/Edit 受限于 cwd，add_dirs 天然只读。**

```
bypassPermissions + disallowed_tools=[]

Read/Glob/Grep  ✅ 读文件、搜索代码
Bash            ✅ 跑测试、装包、git 操作
WebFetch/Search ✅ 查资料
Write           ✅ 可写入 cwd（受限于工作目录）
Edit            ✅ 可编辑 cwd 内文件（受限于工作目录）
```

**常见调整**：

```python
# 偶尔需要改代码：临时放行
run_agent("修复 bug", disallowed_tools=[])

# 连 Bash 都不想让它跑：追加禁用
run_agent("分析代码", disallowed_tools=["Write","Edit","Bash"])

# 只要只读，什么都不让干
run_agent("审查代码", tools=["Read","Glob","Grep"])

# 纯问答
run_agent("解释概念", tools=[])
```

In [ ]:
# 默认配置：所有工具自动放行
# 不需要手动指定 allowed_tools
result = run_agent(
    prompt="列出当前目录下的所有 .py 文件,简要说明每个文件的用途。",
    cwd=r"d:\Admin\Desktop\project\stats_quant_fund",
)

## 4. 自定义工具 (MCP Server 方式)

自定义工具的正确做法:
1. `@sdk.tool(name, description, input_schema)` 定义工具
2. `create_sdk_mcp_server(name=..., tools=[...])` 创建 in-process MCP 服务器
3. `ClaudeAgentOptions(mcp_servers={...})` 注入

> `@sdk.tool` 的参数名是 `input_schema`,不是 `parameters`。
> 工具函数需返回 `{"content": [{"type": "text", "text": "..."}]}` 格式。

In [ ]:
import math, json
import claude_agent_sdk as sdk

# 1. 定义工具
@sdk.tool(
    "calculate_portfolio_metrics",
    "计算投资组合的年化收益率、年化波动率、夏普比率",
    {
        "type": "object",
        "properties": {
            "daily_returns": {
                "type": "array",
                "items": {"type": "number"},
                "description": "日收益率序列(小数形式,如 0.01=1%)"
            },
            "risk_free_rate": {
                "type": "number",
                "description": "无风险利率(年化,默认 0.02)"
            }
        },
        "required": ["daily_returns"]
    }
)
async def calculate_portfolio_metrics(args):
    '''工具实现: 计算组合指标'''
    rets = args["daily_returns"]
    rf = args.get("risk_free_rate", 0.02)

    if not rets:
        return {
            "content": [{"type": "text", "text": "Error: empty returns"}],
            "is_error": True,
        }

    n = len(rets)
    avg = sum(rets) / n
    annual_return = avg * 252

    # 总体方差(无偏估计)
    variance = sum((r - avg) ** 2 for r in rets) / (n - 1)
    annual_vol = math.sqrt(variance * 252)

    sharpe = ((annual_return - rf) / annual_vol) if annual_vol > 0 else 0

    result = {
        "样本数": n,
        "年化收益率": f"{annual_return:.4%}",
        "年化波动率": f"{annual_vol:.4%}",
        "夏普比率": f"{sharpe:.2f}",
    }
    return {"content": [{"type": "text", "text": str(result)}]}


# 2. 创建 MCP 服务器
mcp_config = sdk.create_sdk_mcp_server(
    name="quant_tools",
    tools=[calculate_portfolio_metrics],
)
print(f"MCP Server 已创建: {mcp_config['name']}")


In [ ]:
# 3. 注入 MCP 服务器并运行
result = run_agent(
    prompt=(
        "某策略最近5个交易日的日收益率分别为: "
        "0.008, -0.003, 0.012, -0.005, 0.006。\n"
        "请使用 calculate_portfolio_metrics 工具计算年化收益率和夏普比率。"
    ),
    system_prompt="你是量化分析助手。计算指标时使用 calculate_portfolio_metrics 工具。",
    mcp_servers={"quant": mcp_config},
    max_turns=5,
)

## 5. 生产级模板

以下模板可直接复用到你的项目中。

In [ ]:
'''
Claude Agent SDK 生产级模板
=============================

默认策略: 全部放行。Write/Edit 受限于 cwd（.sdk_sessions/），add_dirs 天然只读。
模型可以读文件、跑命令、查资料，也能在 cwd 中写文件，但不能改 add_dirs 中的代码。

使用方式:
  # cc_sdk.py 在 agent/ 目录下
  import sys
  sys.path.insert(0, r"D:\Admin\Desktop\project\stats_quant_fundgent")
  from cc_sdk import inject_config, run_agent
  inject_config()

  # 日常使用：什么都不用配
  result = run_agent(
      prompt="分析这个项目的代码结构，看看有没有可以优化的地方",
      cwd="d:/your/project",
  )

  # 需要限制写入：只读模式
  result = run_agent(
      prompt="审查这个模块的安全性",
      tools=["Read", "Glob", "Grep"],  # 只读
      cwd="d:/your/project",
  )

  # 纯问答
  result = run_agent("解释夏普比率", tools=[])

  # 限制轮次
  result = run_agent("一句话总结", max_turns=3)

  # 带 skills
  result = run_agent(
      prompt="帮我审查并修复代码",
      skills="all",
      effort="xhigh",
      disallowed_tools=[],  # 默认已放开
  )

  # 获取结果
  print(result["text"])          # 完整回复文本
  print(result["duration_ms"])   # 耗时(ms)
  print(result["num_turns"])     # 对话轮次
  print(result.get("cost"))      # USD 费用

故障排查:
  - "CLIConnectionError": 检查 ANTHROPIC_BASE_URL 和
    ANTHROPIC_AUTH_TOKEN 是否正确注入
'''

# 综合示例
result = run_agent(
    prompt="用一句话解释什么是量化交易,再列出3个常见策略类型。",
    system_prompt="量化金融专家。简洁、专业。",
)
